In [139]:
import numpy as np
import pandas as pd

In [140]:
books = pd.read_csv('books.csv')
users = pd.read_csv('users.csv')
ratings = pd.read_csv('ratings.csv')

In [141]:
books['Book-Title'][0]

'TINH GIAP HON TUONG'

In [142]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [143]:
ratings.head()

,User-ID,Book-Id,Book-Rating
0,1,16,0.0
1,2,24,0.0
2,2,19,0.0
3,2,20,10.0
4,3,6,0.0


In [144]:
print(books.shape)
print(ratings.shape)
print(users.shape)

(39, 5)
(501, 3)
(100, 3)


In [145]:
books.isnull().sum()

Book-Id                0
Book-Title             0
Book-Author            0
Year-Of-Publication    0
Publisher              0
dtype: int64

In [146]:
users.isnull().sum()

User-ID      0
Location     0
Age         46
dtype: int64

In [147]:
ratings.isnull().sum()

User-ID        0
Book-Id        0
Book-Rating    1
dtype: int64

In [148]:
books.duplicated().sum()

0

In [149]:
ratings.duplicated().sum()

7

In [150]:
users.duplicated().sum()

0

## Hệ thống gợi ý dựa trên mức độ phổ biến

In [151]:
ratings_with_name = ratings.merge(books,on='Book-Id')

In [152]:
num_rating_df = ratings_with_name.groupby('Book-Title').count()['Book-Rating'].reset_index()
num_rating_df.rename(columns={'Book-Rating':'num_ratings'},inplace=True)
num_rating_df

,Book-Title,num_ratings
0,ANT-MAN PRELUDE,13
1,BLUE ARCHIVE! (GLOBAL),15
2,BLUE BOX,14
3,CHUYEN SINH THANH THAT HOANG TU,12
4,CONG TU BIET TU,17
5,CUONG MA TAI THE,14
6,DAI CHU TIEN LAI,13
7,DAI PHUNG DA CANH NHAN,9
8,DAU LA DAI LUC,12
9,DU BAO KHAI HUYEN,16


In [153]:
avg_rating_df = ratings_with_name.groupby('Book-Title').mean()['Book-Rating'].reset_index()
avg_rating_df.rename(columns={'Book-Rating':'avg_rating'},inplace=True)
avg_rating_df

,Book-Title,avg_rating
0,ANT-MAN PRELUDE,3.846154
1,BLUE ARCHIVE! (GLOBAL),3.466667
2,BLUE BOX,4.142857
3,CHUYEN SINH THANH THAT HOANG TU,4.666667
4,CONG TU BIET TU,5.352941
5,CUONG MA TAI THE,4.928571
6,DAI CHU TIEN LAI,4.692308
7,DAI PHUNG DA CANH NHAN,4.444444
8,DAU LA DAI LUC,3.666667
9,DU BAO KHAI HUYEN,3.250000


In [154]:
popular_df = num_rating_df.merge(avg_rating_df,on='Book-Title')
popular_df

,Book-Title,num_ratings,avg_rating
0,ANT-MAN PRELUDE,13,3.846154
1,BLUE ARCHIVE! (GLOBAL),15,3.466667
2,BLUE BOX,14,4.142857
3,CHUYEN SINH THANH THAT HOANG TU,12,4.666667
4,CONG TU BIET TU,17,5.352941
5,CUONG MA TAI THE,14,4.928571
6,DAI CHU TIEN LAI,13,4.692308
7,DAI PHUNG DA CANH NHAN,9,4.444444
8,DAU LA DAI LUC,12,3.666667
9,DU BAO KHAI HUYEN,16,3.250000


In [155]:
popular_df = popular_df[popular_df['num_ratings']>=10].sort_values('avg_rating',ascending=False).head(50)

In [156]:
popular_df = popular_df.merge(books,on='Book-Title').drop_duplicates('Book-Title')[['Book-Title','Book-Author','num_ratings','avg_rating']]

In [157]:
popular_df

,Book-Title,Book-Author,num_ratings,avg_rating
0,NGUOI LUN | DWARVES,Richard North Patterson,19,5.473684
1,CONG TU BIET TU,Robynn Clairday,17,5.352941
2,TINH GIAP HON TUONG,Mark P. O. Morford,28,5.000000
4,CUONG MA TAI THE,Stephan Jaramillo,14,4.928571
5,TRIGUN MAXIMUM,Amy Tan,18,4.777778
6,XUAN THU BA ?O,Scott Turow,13,4.769231
7,DAI CHU TIEN LAI,Mordecai Richler,13,4.692308
8,CHUYEN SINH THANH THAT HOANG TU,David Cordingly,12,4.666667
9,QUAI VAT GUI,Mary-Kate &amp; Ashley Olsen,14,4.642857
10,SOLO LEVELING,Harper Lee,15,4.466667


In [158]:
popular_df['Book-Author'][0]

'Richard North Patterson'

## Hệ thống đề xuất dựa trên Collaborative Filtering

In [159]:
x = ratings_with_name.groupby('User-ID').count()['Book-Rating'] > 6
padhe_likhe_users = x[x].index

In [160]:
x

User-ID
1      False
2      False
3      False
4      False
5      False
       ...  
96     False
97     False
98     False
99      True
100     True
Name: Book-Rating, Length: 99, dtype: bool

In [161]:
filtered_rating = ratings_with_name[ratings_with_name['User-ID'].isin(padhe_likhe_users)]

In [162]:
filtered_rating

,User-ID,Book-Id,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher
1,8,16,0.0,LEN CAP HOC VIEN HIEP SI,Loren D. Estleman,1998,Brilliance Audio - Trade
3,17,16,0.0,LEN CAP HOC VIEN HIEP SI,Loren D. Estleman,1998,Brilliance Audio - Trade
12,53,16,8.0,LEN CAP HOC VIEN HIEP SI,Loren D. Estleman,1998,Brilliance Audio - Trade
14,69,16,0.0,LEN CAP HOC VIEN HIEP SI,Loren D. Estleman,1998,Brilliance Audio - Trade
20,17,24,0.0,QUAI VAT GUI,Mary-Kate &amp; Ashley Olsen,2000,HarperEntertainment
...,...,...,...,...,...,...,...
484,56,30,9.0,VO NGHICH,C.S. Lewis,1996,Scribner
488,87,30,8.0,VO NGHICH,C.S. Lewis,1996,Scribner
493,44,39,0.0,ONE PIECE,LAURA HILLENBRAND,2002,Ballantine Books
496,60,39,0.0,ONE PIECE,LAURA HILLENBRAND,2002,Ballantine Books


In [163]:
y = filtered_rating.groupby('Book-Title').count()['Book-Rating']>=2
famous_books = y[y].index

In [164]:
y

Book-Title
ANT-MAN PRELUDE                      True
BLUE ARCHIVE! (GLOBAL)               True
BLUE BOX                             True
CHUYEN SINH THANH THAT HOANG TU      True
CONG TU BIET TU                      True
CUONG MA TAI THE                     True
DAI CHU TIEN LAI                     True
DAI PHUNG DA CANH NHAN               True
DAU LA DAI LUC                       True
DU BAO KHAI HUYEN                    True
GANNIBAL                             True
GINTAMA                              True
HUYET NGHIEP KI SI CHUYEN SINH       True
KAMITACHI NI HIROWARETA OTOKO        True
LAC LOI PHUT TAN CUNG                True
LAM SIEU SAO TU 0 TUOI               True
LEN CAP HOC VIEN HIEP SI             True
MAT THE PHAM NHAN                    True
MONSTER STEIN                        True
NGUOI LUN | DWARVES                  True
NGUYEN LAI TA LA TU TIEN DAI LAO     True
ONE PIECE                            True
ONEPUNCH MAN                         True
QUAI VAT GUI           

In [165]:
final_ratings = filtered_rating[filtered_rating['Book-Title'].isin(famous_books)]

In [166]:
final_ratings

,User-ID,Book-Id,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher
1,8,16,0.0,LEN CAP HOC VIEN HIEP SI,Loren D. Estleman,1998,Brilliance Audio - Trade
3,17,16,0.0,LEN CAP HOC VIEN HIEP SI,Loren D. Estleman,1998,Brilliance Audio - Trade
12,53,16,8.0,LEN CAP HOC VIEN HIEP SI,Loren D. Estleman,1998,Brilliance Audio - Trade
14,69,16,0.0,LEN CAP HOC VIEN HIEP SI,Loren D. Estleman,1998,Brilliance Audio - Trade
20,17,24,0.0,QUAI VAT GUI,Mary-Kate &amp; Ashley Olsen,2000,HarperEntertainment
...,...,...,...,...,...,...,...
484,56,30,9.0,VO NGHICH,C.S. Lewis,1996,Scribner
488,87,30,8.0,VO NGHICH,C.S. Lewis,1996,Scribner
493,44,39,0.0,ONE PIECE,LAURA HILLENBRAND,2002,Ballantine Books
496,60,39,0.0,ONE PIECE,LAURA HILLENBRAND,2002,Ballantine Books


In [167]:
pt = final_ratings.pivot_table(index='Book-Id',columns='User-ID',values='Book-Rating')

In [168]:
pt.fillna(0,inplace=True)

In [169]:
pt

User-ID,8,10,12,13,14,17,20,26,39,44,53,56,60,67,69,76,85,87,99,100
Book-Id,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,10.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.000000,8.0,10.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,3.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,7.000000,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
6,5.0,0.0,0.0,0.0,0.0,4.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,4.5,0.0,8.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
8,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.000000,0.0,0.0,3.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,9.0,0.0


In [170]:
from sklearn.metrics.pairwise import cosine_similarity

In [171]:
similarity_scores = cosine_similarity(pt)

In [172]:
similarity_scores.shape

(38, 38)

In [173]:
pt.index[0] == 1

True

In [174]:
def recommend(book_id):
    book_id = int(book_id)
    # Kiểm tra xem book_name có tồn tại trong pt.index không
    if book_id in pt.index:
        # index fetch
        index = np.where(pt.index == book_id)[0][0]
        similar_items = sorted(list(enumerate(similarity_scores[index])), key=lambda x: x[1], reverse=True)[1:5]
        
        data = []
        for i in similar_items:
            item = []
            temp_df = books[books['Book-Id'] == pt.index[i[0]]]
            item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
            item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
            item.extend(list(temp_df.drop_duplicates('Book-Title')['Year-Of-Publication'].values))
            
            data.append(item)
        
        return data
    else:
        print("Book not found in database")
        return None


In [175]:
recommend(1)

[['TA LA DAI THAN TIEN', 'Julia Oliver', 1994],
 ['Tanbo de Hirotta Onna Kishi', 'R. J. Kaiser', 1999],
 ['DAI CHU TIEN LAI', 'Mordecai Richler', 2002],
 ['SCRALIA E-W', 'Charlotte Link', 1991]]

In [176]:
pt.index[32]

34

In [177]:
import pickle
pickle.dump(popular_df,open('popular.pkl','wb'))

In [178]:
books.drop_duplicates('Book-Title')

,Book-Id,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,1,TINH GIAP HON TUONG,Mark P. O. Morford,2002,Oxford University Press
1,2,GANNIBAL,Richard Bruce Wright,2001,HarperFlamingo Canada
2,3,MONSTER STEIN,Carlo D'Este,1991,HarperPerennial
3,4,LAC LOI PHUT TAN CUNG,Gina Bari Kolata,1999,Farrar Straus Giroux
4,5,KAMITACHI NI HIROWARETA OTOKO,E. J. W. Barber,1999,W. W. Norton &amp; Company
5,6,TRIGUN MAXIMUM,Amy Tan,1991,Putnam Pub Group
6,7,SIEU CAP THAN CO NHAN,Robert Cowley,2000,Berkley Publishing Group
7,8,XUAN THU BA ?O,Scott Turow,1993,Audioworks
8,9,CHUYEN SINH THANH THAT HOANG TU,David Cordingly,1996,Random House
9,10,DAU LA DAI LUC,Ann Beattie,2002,Scribner


In [179]:
pickle.dump(pt,open('pt.pkl','wb'))
pickle.dump(books,open('books.pkl','wb'))
pickle.dump(similarity_scores,open('similarity_scores.pkl','wb'))